# Predicting bioactivity of SARS-CoV-2 3C-like proteinase inhibitors

Machine-learning classifiers that separate active from inactive 3CLpro inhibitors, using IC50 data from ChEMBL and RDKit Morgan fingerprints.

**Workflow:** retrieve, clean and label, featurize, train, evaluate, predict.
Run all cells top to bottom. Results depend on the live ChEMBL release, so your numbers may differ slightly between runs.

In [ ]:
# Run once if the packages are missing
# %pip install rdkit scikit-learn xgboost lightgbm seaborn matplotlib pandas numpy chembl_webresource_client joblib

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score, precision_score,
                             recall_score, roc_curve, confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sns.set_theme(style="whitegrid")
SEED = 42
TARGET_ID = "CHEMBL4523582"   # SARS-CoV-2 3C-like proteinase; verify on chembl.org
ACTIVE_NM, INACTIVE_NM = 1000, 10000
RADIUS, N_BITS = 2, 2048

## 1. Retrieve IC50 records from ChEMBL

In [ ]:
from chembl_webresource_client.new_client import new_client

records = new_client.activity.filter(target_chembl_id=TARGET_ID, standard_type="IC50").only(
    ["canonical_smiles", "standard_value", "standard_units", "standard_relation"])
raw = pd.DataFrame(list(records))
raw.to_csv("chembl_3clpro_ic50_raw.csv", index=False)   # cache so you need not re-query
print(raw.shape)
raw.head()

## 2. Clean and label
Keep exact measurements in nM, average duplicate molecules, then label by IC50. Molecules between the two cutoffs are dropped as ambiguous.

In [ ]:
df = raw.dropna(subset=["canonical_smiles", "standard_value"]).copy()
df = df[(df["standard_units"] == "nM") & (df["standard_relation"] == "=")]
df["ic50_nM"] = pd.to_numeric(df["standard_value"], errors="coerce")
df = df[df["ic50_nM"] > 0].rename(columns={"canonical_smiles": "smiles"})
df = df.groupby("smiles", as_index=False)["ic50_nM"].median()

df = df[(df["ic50_nM"] <= ACTIVE_NM) | (df["ic50_nM"] >= INACTIVE_NM)].copy()
df["active"] = (df["ic50_nM"] <= ACTIVE_NM).astype(int)
df = df.reset_index(drop=True)
print(df["active"].value_counts().rename({1: "active", 0: "inactive"}))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sns.histplot(np.log10(df["ic50_nM"]), bins=30, hue=df["active"].map({1: "active", 0: "inactive"}), ax=ax)
ax.set_xlabel("log10 IC50 (nM)")
plt.show()

## 3. Molecular properties by class

In [ ]:
def props(smi):
    m = Chem.MolFromSmiles(smi)
    return pd.Series({"Molecular weight": Descriptors.MolWt(m), "LogP": Descriptors.MolLogP(m),
                      "H-bond donors": Descriptors.NumHDonors(m), "H-bond acceptors": Descriptors.NumHAcceptors(m)}) if m else pd.Series(dtype=float)

P = df["smiles"].apply(props)
P["Class"] = df["active"].map({1: "active", 0: "inactive"})
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, [c for c in P.columns if c != "Class"]):
    sns.boxplot(data=P, x="Class", y=col, ax=ax)
plt.tight_layout()
plt.show()

## 4. Featurize with Morgan fingerprints

In [ ]:
gen = rdFingerprintGenerator.GetMorganGenerator(radius=RADIUS, fpSize=N_BITS)
X, keep = [], []
for i, s in enumerate(df["smiles"]):
    m = Chem.MolFromSmiles(s)
    if m is not None:
        X.append(gen.GetFingerprintAsNumPy(m))
        keep.append(i)
X = np.array(X, dtype=np.uint8)
y = df["active"].iloc[keep].to_numpy()
print(X.shape, y.mean().round(3), "fraction active")

## 5. Train and evaluate
Stratified 80/20 split so both classes appear in train and test.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=SEED),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, eval_metric="logloss", random_state=SEED),
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.1, random_state=SEED, verbose=-1),
}
rows, curves, probs = [], {}, {}
for name, model in models.items():
    model.fit(Xtr, ytr)
    p = model.predict_proba(Xte)[:, 1]
    pred = (p >= 0.5).astype(int)
    probs[name], curves[name] = p, roc_curve(yte, p)
    rows.append({"Model": name, "AUC-ROC": roc_auc_score(yte, p), "F1": f1_score(yte, pred),
                 "Accuracy": accuracy_score(yte, pred), "Precision": precision_score(yte, pred),
                 "Recall": recall_score(yte, pred)})
results = pd.DataFrame(rows).set_index("Model")
results.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for name, (fpr, tpr, _) in curves.items():
    ax.plot(fpr, tpr, label=f"{name} (AUC {results.loc[name, 'AUC-ROC']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, (name, p) in zip(axes, probs.items()):
    sns.heatmap(confusion_matrix(yte, (p >= 0.5).astype(int)), annot=True, fmt="d", cbar=False,
                xticklabels=["inactive", "active"], yticklabels=["inactive", "active"], ax=ax)
    ax.set_title(name); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 6. Which fingerprint bits matter most
Bits are hashed substructures, so a bit index is not a fixed chemical group, but the ranking shows how concentrated the signal is.

In [ ]:
imp = pd.Series(models["Random Forest"].feature_importances_).nlargest(15)
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=imp.values, y=imp.index.astype(str), color="#0E8A8C", ax=ax)
ax.set_xlabel("Random Forest importance"); ax.set_ylabel("Bit index")
plt.show()

## 7. Score a new molecule

In [ ]:
def predict_active(smiles):
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        raise ValueError("Could not parse SMILES")
    x = gen.GetFingerprintAsNumPy(m).reshape(1, -1)
    return {name: round(float(model.predict_proba(x)[0, 1]), 3) for name, model in models.items()}

predict_active("CC(C)C[C@H](NC(=O)OCc1ccccc1)C(=O)N")

## 8. Notes
After running, add your own summary here: which model scored best on AUC-ROC and F1, how the classes were balanced, and any limits (for example, label noise from mixing assay types, or random rather than scaffold-based splits, which can overestimate performance on new chemical series).